![DSPy](/workspace/src/assets/dspy_logo.png)

DSPy (Declarative Self-improving Python) es un framework para programar aplicaciones basadas en modelos de lenguaje. En lugar de centrar el desarrollo en escribir y ajustar prompts manualmente, DSPy permite definir qué debe hacer el sistema, cómo medir si lo hace correctamente y utilizar esa información para optimizar el programa automáticamente.

![alt text](assets/promt_vs_dspy.png)

## Flujos de trabajo

### Prompt

![alt text](assets/w_prompt.png)

### DSPY

![alt text](assets/w_dspy.png)

![alt text](assets/w_dspy_2.png)

____

## Set up

In [ ]:
# import dspy

# lm = dspy.LM("modelo_que_queremos_usar", api_key="YOUR_OPENAI_API_KEY")

# dspy.configure(lm=lm)

In [1]:
import os
import importlib
import lm_config as dspy_lm_config

from lm_config import configure_lm
import dspy


os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"
os.environ["OLLAMA_MODEL"] = "llama3.2:latest"
# os.environ["OLLAMA_MODEL"] = "phi3:latest"

importlib.reload(dspy_lm_config)


lm = configure_lm(max_tokens=1024)
print(lm.model)

ollama_chat/llama3.2:latest


_____

## 01 - Signatures - Que quiero hacer

Una <strong style="color:#c2410c">Signature</strong> se puede entender como una tarea que quiero realizar describiendo la a travez de sus entradas y salidas.
La idea es definir una tarea simple:
- input → pregunta
- output → respuesta

DSPy se encargará de construir internamente el prompting necesario.

![alt text](assets/signature.png)

In [2]:
texto = "ingles -> espanol"
traducir = dspy.Predict(texto)
resultado = traducir(ingles="The security deposit equals two months of rent.")
print(resultado.espanol)

El depósito de seguridad equivale a dos meses de alquiler.


In [3]:
reescribir = dspy.Predict("texto: str, tono: str -> reescritura: str")
r = reescribir(
    texto="hay un recargo en caso de no pagar a tiempo",
    tono="formal y educado",
)
print(r.reescritura)

se aplicará un recargo en el caso de no pagar a tiempo


## 02 - Class-based Signatures

Las String Signatures son muy cómodas para comenzar, pero rápidamente se quedan cortas cuando queremos agregar instrucciones, descripciones o tipos más complejos. Para resolver esto, DSPy ofrece las <strong style="color:#c2410c">Class-based Signatures</strong>.

In [4]:
class DocumentSummary(dspy.Signature):
    """Resume un documento para una audiencia determinada."""

    documento: str = dspy.InputField()
    audiencia: str = dspy.InputField()
    max_palabras: int = dspy.InputField()

    resumen: str = dspy.OutputField()
    titulo: str = dspy.OutputField()
    palabras_clave: list[str] = dspy.OutputField()

In [5]:
summarizer = dspy.Predict(DocumentSummary)

In [6]:
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones con modelos de lenguaje
    utilizando Signatures y Modules en lugar de prompts escritos manualmente.
    """,
    audiencia="Desarrolladores Python principiantes",
    max_palabras=50
)

In [7]:
print(resultado.titulo)
print(resultado.resumen)
print(resultado.palabras_clave)

Introducción a DSPy
DSPy es un framework que permite construir aplicaciones con modelos de lenguaje utilizando Signatures y Modules, lo que facilita la creación de aplicaciones sin necesidad de escribir prompts manualmente.
['DSPy', 'framework', 'modelos de lenguaje', 'Signatures', 'Modules']


![alt text](assets/class-base-signatu.png)

In [8]:
from typing import Literal

Audiencia = Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]

class DocumentSummary(dspy.Signature):
    """
    Resume un documento según la audiencia.
    """
    documento: str = dspy.InputField()
    audiencia: Audiencia = dspy.InputField()
    resumen: str = dspy.OutputField()
    titulo: str = dspy.OutputField()

In [9]:
summarizer = dspy.Predict(DocumentSummary)

In [10]:

resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="principiantes",
)

print(resultado.titulo)
print(resultado.resumen)

Introducción a DSPy
DSPy es una plataforma para desarrollar aplicaciones utilizando Signatures y Modules, lo que permite una mayor flexibilidad y personalización en el proceso de creación.


In [11]:

resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="ejecutivos",
)

print(resultado.titulo)
print(resultado.resumen)

Construyendo Aplicaciones con DSPy a Través de Signatures y Modules
El documento DSPy proporciona una forma flexible de construir aplicaciones mediante el uso de Signatures y Modules.


### 02.1- Datos estructurados

Muchas aplicaciones necesitan devolver información más compleja. En lugar de devolver varios campos independientes, podemos agruparlos en un único objeto utilizando <strong style="color:#c2410c">Pydantic</strong>. Esto hace que nuestras `Signatures` sean más expresivas y que el resultado sea mucho más fácil de utilizar desde Python.

In [12]:
from pydantic import BaseModel

class ResumenDocumento(BaseModel):
    titulo: str
    resumen: str
    palabras_clave: list[str]


class DocumentSummary(dspy.Signature):
    """Resume un documento según la audiencia."""
    documento: str = dspy.InputField()
    audiencia: Audiencia = dspy.InputField()
    resultado: ResumenDocumento = dspy.OutputField()

In [13]:
summarizer = dspy.Predict(DocumentSummary)

In [14]:
resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="principiantes",
)

print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)


Resumen del documento
El documento describe cómo se pueden construir aplicaciones utilizando DSPy, una herramienta de desarrollo que permite la creación de aplicaciones utilizando firmas y módulos.
['DSPy', 'aplicaciones', 'Signatures', 'Modules']


In [15]:
resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="ejecutivos",
)

print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)


Características de DSPy
DSPy es una herramienta para construir aplicaciones utilizando Signatures y Modules.
['aplicaciones', 'Signatures', 'Modules']


## 03 - Modules - Como lo quiero hacer

Los <strong style="color:#c2410c">Modules</strong> implementan distintas estrategias para ejecutar una Signature. Podemos mantener la misma tarea y cambiar el Module para modificar la forma en que el modelo intenta resolverla.

![alt text](assets/Modules.png)

In [16]:
from typing import Literal
from pydantic import BaseModel, Field

CLAUSULA = (
    "El depósito de garantía equivaldrá a tres meses de alquiler. "
    "El propietario podrá ingresar a la vivienda sin aviso previo. "
    "Las expensas ordinarias y extraordinarias estarán a cargo del inquilino. "
    "El contrato podrá rescindirse unilateralmente con 5 días de preaviso."
)

TemaClausula = Literal["deposito", "ingreso", "expensas", "rescision", "otro"]
NivelRiesgo = Literal["bajo", "medio", "alto"]
Severidad = Literal["leve", "moderada", "severa"]

### a. `dspy.Predict` — Respuesta directa

Es el módulo más básico de DSPy.

Recibe los datos de entrada definidos en una Signature, realiza una llamada al LLM y devuelve los campos de salida esperados.

**Características:**
- Genera respuestas directamente.
- No agrega automáticamente pasos intermedios de razonamiento.
- No utiliza herramientas externas por sí mismo.
- Es útil para tareas sencillas como clasificación, extracción de información y preguntas y respuestas.

In [17]:
class ClasificarClausula(dspy.Signature):
    """Clasifica el tema principal de una cláusula de alquiler."""

    clausula: str = dspy.InputField()
    tema: TemaClausula = dspy.OutputField()

clasificar = dspy.Predict(ClasificarClausula)
r = clasificar(clausula=CLAUSULA)
print(r.tema)

otro


In [18]:
r = clasificar(
    clausula="El depósito de garantía equivaldrá a tres meses de alquiler."
)
print(r.tema)

deposito


### b. `dspy.ChainOfThought` — Razonamiento

Implementa la técnica **Chain of Thought** (cadena de pensamiento).

Solicita al LLM que genere un razonamiento antes de producir la respuesta final.

**Características:**
- Agrega automáticamente un campo `reasoning` a la Signature.
- Genera un razonamiento previo a la respuesta.
- Es útil para tareas que requieren análisis, interpretación o resolución de problemas.

In [19]:
class EvaluacionRiesgo(BaseModel):
    nivel_riesgo: NivelRiesgo
    explicacion: str = Field(
        description="2-3 frases claras para el inquilino, sin repetir el razonamiento interno"
    )
    senales: list[str] = Field(
        description="Lista concreta de alertas, ej: 'ingreso sin aviso', 'depósito de 3 meses'"
    )

class EvaluarRiesgoInquilino(dspy.Signature):
    """Evalúa el riesgo de la cláusula para el inquilino."""

    clausula: str = dspy.InputField()
    resultado: EvaluacionRiesgo = dspy.OutputField()

evaluar = dspy.ChainOfThought(EvaluarRiesgoInquilino)
r = evaluar(clausula=CLAUSULA)
print("razonamiento:", r.reasoning)
print("riesgo:", r.resultado.nivel_riesgo)
print("explicación:", r.resultado.explicacion)
print("señales:", r.resultado.senales)

razonamiento: El propietario puede ingresar sin aviso previo y no está claro si se incluyen las expensas ordinarias y extraordinarias, lo que genera incertidumbre sobre el riesgo del contrato.
riesgo: alto
explicación: El contrato presenta algunas cláusulas poco claras y flexibles que podrían generar conflictos entre la parte de alquiler y la del propietario. El depósito de garantía solo cubrirá tres meses de alquiler, lo cual puede ser un riesgo para el inquilino si no es suficiente para cubrir los gastos futuros.
señales: ['ingreso sin aviso previo', 'depósito de garantía limitado', 'incertidumbre sobre gastos']


### c. `dspy.ProgramOfThought` — Ejecución de código

Permite que el LLM genere y ejecute código Python para resolver una tarea.

Es especialmente útil para problemas matemáticos, cálculos estadísticos y operaciones algorítmicas.

**Características:**
- Genera código Python para resolver problemas.
- Ejecuta el código y obtiene sus resultados.
- Puede detectar errores de ejecución e intentar corregirlos.
- Utiliza los resultados de la ejecución para generar la respuesta final.

In [20]:
class AnalisisCostosAlquiler(dspy.Signature):
    """Analiza montos de un contrato de alquiler y calcula métricas pedidas."""

    montos: list[float] = dspy.InputField(
        desc="Lista de montos mensuales o conceptos en pesos"
    )
    metricas_requeridas: list[str] = dspy.InputField(
        desc="Métricas a calcular, ej: ['suma', 'promedio', 'maximo']"
    )
    resultados: dict[str, float] = dspy.OutputField(
        desc="Diccionario con las métricas calculadas"
    )

analizar = dspy.ProgramOfThought(AnalisisCostosAlquiler)

montos = [450000.0, 450000.0, 450000.0]  # 3 meses de depósito
metricas = ["suma", "promedio"]

r = analizar(montos=montos, metricas_requeridas=metricas)
print(r.resultados)
# esperado algo como: {'suma': 1350000.0, 'promedio': 450000.0}

{'suma': 410500000.0, 'promedio': 450000.0}


In [24]:
class CalcularCostoAlquiler(dspy.Signature):
    """Calcula montos del contrato de alquiler ejecutando código."""
    pregunta: str = dspy.InputField()
    respuesta: int = dspy.OutputField(desc="Monto total en pesos, sin símbolos")
pot = dspy.ProgramOfThought(CalcularCostoAlquiler)
r = pot(
    pregunta=(
        "El alquiler es 450000 por mes y el depósito equivaldrá a 3 meses. "
        "¿Cuánto debe pagar el inquilino solo en concepto de depósito?"
    )
)
print(r.respuesta)  # esperado: 1350000

1350000


### d. `dspy.ReAct` — Razonamiento y herramientas

Implementa la técnica **Reasoning and Acting** (razonar y actuar).

Permite que el LLM utilice herramientas externas para obtener información o realizar acciones.

**Características:**
- Combina razonamiento con ejecución de herramientas.
- Puede utilizar herramientas como buscadores, funciones o APIs.
- Observa los resultados de las herramientas antes de decidir el siguiente paso.
- Repite el proceso hasta obtener una respuesta o alcanzar un límite de iteraciones.
- Es útil para desarrollar agentes basados en LLMs.

In [21]:
TerminoGlosario = Literal["deposito", "expensas", "rescision", "preaviso"]
def consultar_glosario(termino: TerminoGlosario) -> str:
    """Busca definiciones de términos de contratos de alquiler."""
    glosario: dict[TerminoGlosario, str] = {
        "deposito": "Suma que garantiza el contrato; suele devolverse al finalizar si no hay deudas ni daños.",
        "expensas": "Gastos del edificio. Ordinarias = mantenimiento habitual; extraordinarias = obras mayores.",
        "rescision": "Finalización anticipada del contrato por una de las partes.",
        "preaviso": "Aviso previo obligatorio antes de terminar o modificar el contrato.",
    }
    return glosario.get(termino, "Término no encontrado en el glosario.")
class AsistenteAlquiler(dspy.Signature):
    """Responde dudas sobre cláusulas de alquiler usando el glosario si hace falta."""
    pregunta: str = dspy.InputField()
    respuesta: str = dspy.OutputField()
agente = dspy.ReAct(AsistenteAlquiler, tools=[consultar_glosario])
r = agente(
    pregunta="Según la cláusula, ¿qué significa que el depósito sea de tres meses?"
)
print(r.respuesta)

La cantidad que debe pagar el inquilino cuando termina su contrato de alquiler es conocida como "depósito" y generalmente es igual al equivalente en tres meses de los pagos mensuales del alquiler.


## Modules y helpers adicionales

### f. `dspy.MultiChainComparison` — Comparación de razonamientos

Permite comparar múltiples razonamientos generados previamente para producir una respuesta final.

**Características:**
- Recibe diferentes predicciones o razonamientos.
- Utiliza un LLM para comparar los resultados.
- Genera una respuesta final a partir de la información proporcionada.

In [25]:
class InterpretarClausula(dspy.Signature):
    """Identifica el punto más riesgoso para el inquilino."""

    clausula: str = dspy.InputField()
    tema_riesgoso: TemaClausula = dspy.OutputField()
    punto_riesgoso: str = dspy.OutputField(desc="Una frase en español")

# 3 lecturas distintas (como si vinieran de 3 corridas)
completions = [
    dspy.Prediction(
        reasoning="El depósito de 3 meses es una carga alta al inicio.",
        tema_riesgoso="deposito",
        punto_riesgoso="El depósito de tres meses exige mucho capital inicial.",
    ),
    dspy.Prediction(
        reasoning="Entrar sin aviso afecta la privacidad del inquilino.",
        tema_riesgoso="ingreso",
        punto_riesgoso="El propietario puede entrar sin aviso previo.",
    ),
    dspy.Prediction(
        reasoning="Las expensas extraordinarias pueden ser muy costosas.",
        tema_riesgoso="expensas",
        punto_riesgoso="El inquilino paga expensas ordinarias y extraordinarias.",
    ),
]

for i, c in enumerate(completions, 1):
    print(f"Intento {i}: [{c.tema_riesgoso}] {c.punto_riesgoso}")

comparar = dspy.MultiChainComparison(InterpretarClausula, M=3)
final = comparar(completions, clausula=CLAUSULA)
print("Lectura final:", final.tema_riesgoso, "-", final.punto_riesgoso)

Intento 1: [deposito] El depósito de tres meses exige mucho capital inicial.
Intento 2: [ingreso] El propietario puede entrar sin aviso previo.
Intento 3: [expensas] El inquilino paga expensas ordinarias y extraordinarias.
Lectura final: otro - La presencia de cláusulas contractuales como el depósito de garantía y la posibilidad de rescisión unilateral pueden ser puntos de riesgo para los inquilinos, ya que pueden generar situaciones inciertas y financieramente desafiantes.


### g. `majority` — Votación mayoritaria

Es una función auxiliar (*helper*) que permite seleccionar la respuesta más frecuente entre múltiples predicciones.

**Características:**
- Recibe varias predicciones.
- Compara las respuestas generadas.
- Selecciona la respuesta que aparece con mayor frecuencia.
- No necesita una llamada adicional al LLM para realizar la votación.

In [26]:
class SeveridadContrato(dspy.Signature):
    """Asigna una severidad global al contrato para el inquilino."""

    clausula: str = dspy.InputField()
    severidad: Severidad = dspy.OutputField()

# Votos distintos (como si vinieran de 5 corridas)
predicciones = [
    dspy.Prediction(severidad="leve"),
    dspy.Prediction(severidad="moderada"),
    dspy.Prediction(severidad="severa"),
    dspy.Prediction(severidad="severa"),
    dspy.Prediction(severidad="severa"),
]

for i, p in enumerate(predicciones, 1):
    print(f"{i}: {p.severidad}")

ganador = dspy.majority(predicciones, field="severidad")
print("Severidad por mayoría:", ganador.severidad)
# esperado: severa  (3 de 5)

1: leve
2: moderada
3: severa
4: severa
5: severa
Severidad por mayoría: severa
